In [ ]:
# === Setup ===
# Runtime: ~5 phút trên Colab T4
# Hardware: CPU ok / Nên dùng GPU
import os, random
import numpy as np
import torch
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

if 'google.colab' in str(get_ipython()):
    print("Running on Google Colab...")
elif 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    print("Running on Kaggle...")
else:
    print("Running locally...")


# Xây dựng Autograd Engine (Micrograd)
Chúng ta sẽ xây dựng class `Value` để theo dõi các phép toán.


In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label
        
    def __repr__(self):
        return f'Value(data={self.data}, grad={self.grad})'
        
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

x = Value(2.0, label='x')
y = Value(3.0, label='y')
z = x + y; z.label = 'z'
z.grad = 1.0
z._backward()
print(x, y)
